# Data Exploration
This notebook loads and performs a preliminary exploration of the CSV datasets located in the `../dataset` directory.

In [117]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from IPython.display import display

# Configure plotting style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [118]:
# Locate all CSV files in the dataset directory
dataset_dir = "../dataset"
csv_pattern = os.path.join(dataset_dir, "*.csv")
csv_files = glob.glob(csv_pattern)

print(f"Found {len(csv_files)} CSV files in '{dataset_dir}':")
for f in csv_files:
    print(f"- {os.path.basename(f)}")

Found 5 CSV files in '../dataset':
- airlines.csv
- cities.csv
- flights_cleaned.csv
- flights_live.csv
- airports.csv


In [119]:
cities = pd.read_csv("../dataset/cities.csv")

In [120]:
airports = pd.read_csv("../dataset/airports.csv")

In [121]:
flights = pd.read_csv("../dataset/flights_cleaned.csv")
flights['ORIGIN_AIRPORT'] = flights['ORIGIN_AIRPORT'].astype(str)
flights['DESTINATION_AIRPORT'] = flights['DESTINATION_AIRPORT'].astype(str)

In [122]:
flights.groupby(["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]).size().sort_values(ascending=True).head(100)

ORIGIN_AIRPORT  DESTINATION_AIRPORT
BNA             MIA                    52
XNA             SFO                    52
EWR             AVP                    53
BGR             LGA                    53
AVP             EWR                    53
                                       ..
MSP             MBS                    79
IAD             ORF                    81
ORD             STC                    82
IAH             ANC                    82
FLL             IAD                    83
Length: 100, dtype: int64

# Rotte totali

In [123]:
# Ottiene le coppie univoche come DataFrame
unique_routes = flights[['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']].drop_duplicates()

# Se vuoi vederle come elenco di tuple Python
tuple_list = list(unique_routes.itertuples(index=False, name=None))
print(len(tuple_list))  # Stampa le prime 10 coppie

4243


In [124]:
# Find flights with origin or destination airports not in airports.csv
missing_airports = flights[
    (~flights['ORIGIN_AIRPORT'].isin(airports['IATA_CODE'])) |
    (~flights['DESTINATION_AIRPORT'].isin(airports['IATA_CODE']))
]

# Get unique airport codes that are missing
missing_origin = set(missing_airports['ORIGIN_AIRPORT'].unique()) - set(airports['IATA_CODE'])
missing_destination = set(missing_airports['DESTINATION_AIRPORT'].unique()) - set(airports['IATA_CODE'])

print(f"Missing origin airports: {sorted(map(str, missing_origin))}")
print(f"Missing destination airports: {sorted(map(str, missing_destination))}")
print(f"\nTotal flights with missing airports: {len(missing_airports)}")

Missing origin airports: []
Missing destination airports: []

Total flights with missing airports: 0


In [61]:
# Voli con cadenza fissa: stessi (AIRLINE, ORIGIN, DESTINATION) ripetuti più volte
fixed_schedule = (
    flights.groupby(["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
    .sort_values("count", ascending=False)
)

display(fixed_schedule)

,ORIGIN_AIRPORT,DESTINATION_AIRPORT,count
4183,SFO,LAX,13744
2481,LAX,SFO,13457
2265,JFK,LAX,12016
2446,LAX,JFK,12015
2369,LAS,LAX,9715
...,...,...,...
2761,MDT,DEN,2
453,BNA,TTN,2
2963,MKE,TTN,2
2018,IAH,BDL,2


In [62]:
flights.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [74]:
print(flights.query("ORIGIN_AIRPORT == 'LAX' and DESTINATION_AIRPORT == 'SFO'").groupby("SCHEDULED_DEPARTURE").size().reset_index(name="count").sort_values("count"))

     SCHEDULED_DEPARTURE  count
469                 2320      1
181                 1206      1
419                 2107      1
418                 2103      1
416                 2058      1
..                   ...    ...
262                 1530    214
4                    600    222
88                   905    247
293                 1635    280
37                   700    489

[470 rows x 2 columns]


In [96]:
flights.groupby(["FLIGHT_NUMBER", "AIRLINE"]).size().reset_index(name="count").sort_values("count", ascending=False)

,FLIGHT_NUMBER,AIRLINE,count
330,64,AS,1660
336,65,AS,1659
318,62,AS,1552
289,55,AS,1336
4623,711,NK,1335
...,...,...,...
18932,4853,WN,1
18965,4873,WN,1
18969,4875,WN,1
18999,4890,WN,1


In [97]:
flights.query("FLIGHT_NUMBER == 64 and AIRLINE == 'AS'")

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
4716,2015,1,1,4,AS,64,N799AS,ANC,JNU,1130,...,1312.0,2.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
7020,2015,1,1,4,AS,64,N799AS,JNU,PSG,1355,...,1439.0,-3.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
8374,2015,1,1,4,AS,64,N799AS,PSG,WRG,1524,...,1535.0,-12.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
9454,2015,1,1,4,AS,64,N799AS,WRG,KTN,1630,...,1702.0,0.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
10553,2015,1,1,4,AS,64,N799AS,KTN,SEA,1743,...,2016.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5811170,2015,12,31,4,AS,64,N768AS,ANC,JNU,1105,...,1258.0,10.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5813493,2015,12,31,4,AS,64,N768AS,JNU,PSG,1344,...,1433.0,3.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5814959,2015,12,31,4,AS,64,N768AS,PSG,WRG,1520,...,1540.0,-5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5815876,2015,12,31,4,AS,64,N768AS,WRG,KTN,1630,...,1704.0,2.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
flights.query("FLIGHT_NUMBER == 1 and AIRLINE == 'AA' and DAY_OF_WEEK == 1").groupby("SCHEDULED_DEPARTURE").size().reset_index(name="count").sort_values("count")

,SCHEDULED_DEPARTURE,count
0,900,44


In [111]:
flights

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5325422,2015,12,31,4,B6,688,N657JB,LAX,BOS,2359,...,753.0,-26.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5325423,2015,12,31,4,B6,745,N828JB,JFK,PSE,2359,...,430.0,-16.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5325424,2015,12,31,4,B6,1503,N913JB,JFK,SJU,2359,...,432.0,-8.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5325425,2015,12,31,4,B6,333,N527JB,MCO,SJU,2359,...,330.0,-10.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


# Join cities and airprots

In [156]:
airports = pd.read_csv("../dataset/airports.csv")
cities = pd.read_csv("../dataset/cities.csv")

In [157]:
airports.shape

(322, 7)

In [158]:
def clean_city_name(df, column_name):
    # Tutto minuscolo
    df[column_name] = df[column_name].str.lower()
    # Rimuove punti (St. -> St)
    df[column_name] = df[column_name].str.replace('.', '', regex=False)
    # Rimuove spazi bianchi all'inizio e alla fine
    df[column_name] = df[column_name].str.strip()
    return df

cities = clean_city_name(cities, 'city')
airports = clean_city_name(airports, 'CITY') # o 'city' se l'hai rinominata

In [159]:
combined_df = pd.merge(cities, airports[['CITY', 'STATE']], left_on=['city', 'state_id'], right_on=['CITY', 'STATE'], how='inner')

# Se vuoi SOLO le colonne originali di cities, il merge le manterrà automaticamente.
# Se vuoi essere sicuro di non avere altro:
result = combined_df[cities.columns]

In [160]:
result.shape

(297, 17)

In [161]:
result.to_csv("../dataset/cities_cleaned.csv", index=False)

In [163]:
airports[~airports['CITY'].isin(result['city'])]

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
5,ACK,Nantucket Memorial Airport,nantucket,MA,USA,41.25305,-70.06018
7,ACV,Arcata Airport,arcata/eureka,CA,USA,40.97812,-124.10862
24,AVP,Wilkes-Barre/Scranton International Airport,wilkes-barre/scranton,PA,USA,41.33815,-75.72427
26,BDL,Bradley International Airport,windsor locks,CT,USA,41.93887,-72.68323
40,BPT,Jack Brooks Regional Airport (Southeast Texas ...,beaumont/port arthur,TX,USA,29.95083,-94.02069
45,BRW,Wiley Post-Will Rogers Memorial Airport,barrow,AK,USA,71.28545,-156.76600
68,CMI,University of Illinois - Willard Airport,champaign/urbana,IL,USA,40.03925,-88.27806
86,DFW,Dallas/Fort Worth International Airport,dallas-fort worth,TX,USA,32.89595,-97.03720
114,FLL,Fort Lauderdale-Hollywood International Airport,ft lauderdale,FL,USA,26.07258,-80.15275
126,GPT,Gulfport-Biloxi International Airport,gulfport-biloxi,MS,USA,30.40728,-89.07009


In [137]:
print(cities['city'].value_counts().head(10))

city
Franklin       30
Clinton        23
Fairview       23
Marion         22
Madison        21
Greenville     20
Georgetown     20
Springfield    19
Salem          19
Clayton        19
Name: count, dtype: int64


In [144]:
cities.query("city == 'Franklin'").shape

(30, 17)

In [182]:
flights = pd.read_csv("../dataset/flights_cleaned.csv")

In [185]:
flights.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
       'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT',
       'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE',
       'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON',
       'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
       'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY'],
      dtype='str')

# colonne da togliere

- TAXI_OUT
- WHEELS_OFF
- WHEELS_ON
- TAXI_IN
- CANCELLATION_REASON
- AIR_SYSTEM_DELAY
- SECURITY_DELAY
- AIRLINE_DELAY
- LATE_AIRCRAFT_DELAY
- WEATHER_DELAY

In [217]:
# Definizione della lista delle colonne da rimuovere
cols_to_remove = [
    "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN", 
    "CANCELLATION_REASON", "AIR_SYSTEM_DELAY", "SECURITY_DELAY", 
    "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY", "TAIL_NUMBER"
]

# Rimozione e salvataggio nel DataFrame
# errors='ignore' evita che il codice si blocchi se una colonna è già stata rimossa
flights.drop(columns=cols_to_remove, errors='ignore', inplace=True)

In [218]:
flights.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED
0,2015,1,1,4,AS,98,ANC,SEA,5,2354.0,-11.0,205.0,194.0,169.0,1448,430,408.0,-22.0,0,0
1,2015,1,1,4,AA,2336,LAX,PBI,10,2.0,-8.0,280.0,279.0,263.0,2330,750,741.0,-9.0,0,0
2,2015,1,1,4,US,840,SFO,CLT,20,18.0,-2.0,286.0,293.0,266.0,2296,806,811.0,5.0,0,0
3,2015,1,1,4,AA,258,LAX,MIA,20,15.0,-5.0,285.0,281.0,258.0,2342,805,756.0,-9.0,0,0
4,2015,1,1,4,AS,135,SEA,ANC,25,24.0,-1.0,235.0,215.0,199.0,1448,320,259.0,-21.0,0,0


In [219]:
flights.columns

Index(['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER',
       'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE',
       'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'SCHEDULED_TIME', 'ELAPSED_TIME',
       'AIR_TIME', 'DISTANCE', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME',
       'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED'],
      dtype='str')

In [221]:
flights.to_csv("../dataset/flights_cleaned.csv", index=False)

In [193]:
airlines = pd.read_csv("../dataset/airlines.csv")
airlines.columns

Index(['IATA_CODE', 'AIRLINE'], dtype='str')

In [194]:
airports = pd.read_csv("../dataset/airports.csv")
airports.columns

Index(['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE',
       'LONGITUDE'],
      dtype='str')

In [195]:
cities = pd.read_csv("../dataset/cities_cleaned.csv")
cities.columns

Index(['city', 'city_ascii', 'state_id', 'state_name', 'county_fips',
       'county_name', 'lat', 'lng', 'population', 'density', 'source',
       'military', 'incorporated', 'timezone', 'ranking', 'zips', 'id'],
      dtype='str')

In [310]:
live = pd.read_csv("../dataset/flights_live.csv")
live.columns

Index(['fr24_id', 'flight', 'callsign', 'lat', 'lon', 'track', 'alt', 'gspeed',
       'vspeed', 'squawk', 'timestamp', 'source', 'hex', 'type', 'reg',
       'painted_as', 'operating_as', 'orig_iata', 'orig_icao', 'dest_iata',
       'dest_icao', 'eta'],
      dtype='str')

In [311]:
print(live.shape)
flights_with_null = live[live['painted_as'].isna()]['flight'].unique()
print(flights_with_null)
live = live.dropna(subset=['painted_as', 'operating_as'])
print(live.shape)

(5771, 22)
<StringArray>
['DL2947', 'DL1930']
Length: 2, dtype: str
(5703, 22)


In [316]:
mapping = live.dropna(subset=['dest_iata']).set_index('flight')['dest_iata'].to_dict()
live['dest_iata'] = live['dest_iata'].fillna(live['flight'].map(mapping))

In [317]:
live.shape

(5703, 22)

In [318]:
live.isna().sum()

fr24_id           0
flight            0
callsign          0
lat               0
lon               0
track             0
alt               0
gspeed            0
vspeed            0
squawk            0
timestamp         0
source            0
hex               0
type              0
reg               0
painted_as        0
operating_as      0
orig_iata         0
orig_icao         0
dest_iata         0
dest_icao         6
eta             241
dtype: int64

In [238]:
cols_to_remove = [
    "fr24_id", "track", "squawk", "source", 
    "hex", "type", "reg", 
    "painted_as", "operating_as", "orig_icao", "dest_icao"
]

# Rimozione e salvataggio nel DataFrame
# errors='ignore' evita che il codice si blocchi se una colonna è già stata rimossa
live.drop(columns=cols_to_remove, errors='ignore', inplace=True)

In [239]:
live.columns

Index(['flight', 'callsign', 'lat', 'lon', 'alt', 'gspeed', 'vspeed',
       'timestamp', 'orig_iata', 'dest_iata', 'eta'],
      dtype='str')

In [240]:
live.to_csv("../dataset/flights_live_cleaned.csv", index=False)

In [241]:
live['flight'].value_counts()

flight
DL274     168
AF636      93
AF158      93
AF84       92
DL120      83
         ... 
WN1309     11
UA844      11
DL1183     10
UA1289      9
WN4904      9
Name: count, Length: 150, dtype: int64

In [227]:
missing_airports_orig = live[~live['orig_iata'].isin(airports['IATA_CODE'])][['orig_iata']]
missing_airports_dest = live[~live['dest_iata'].isin(airports['IATA_CODE'])][['dest_iata']]

missing_airports = pd.concat([missing_airports_orig.rename(columns={'orig_iata': 'iata_code'}),
                              missing_airports_dest.rename(columns={'dest_iata': 'iata_code'})]).drop_duplicates()

In [229]:
missing_airports.head(21)

,iata_code
7,LHR
8,AMS
14,CDG
18,SYD
23,HND
26,MDE
32,SJO
36,NRT
37,FRA
70,ICN
